# 📈 Cookbook: scikit-learn models on ciphertext

The processor is a score provider: it owns the model, fitted on **its own
historical data**, in the clear. The client (data controller) sends only
encrypted records. `pdpg.sklearn.wrap` replays the fitted parameters as
encrypted arithmetic — the model never sees the data, the data never sees
the model file.

In [ ]:
%%time
%pip install -q git+https://github.com/PDPG-lab/pypdpg

In [ ]:
import numpy as np
import pypdpg as pdpg

# the client's side: keys + encrypted records
ctx = pdpg.Context.create()
rng = np.random.default_rng(0)
X_client = rng.normal(size=(100, 4))
X_enc = pdpg.encrypt(X_client, ctx)
X_enc

## The provider's model — plain sklearn, fitted on its own history

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng_p = np.random.default_rng(1)
X_hist = rng_p.normal(size=(1000, 4))
y_hist = (X_hist[:, 0] - X_hist[:, 1] + rng_p.normal(scale=0.3, size=1000) > 0).astype(int)

pipe = make_pipeline(StandardScaler(), LogisticRegression(C=0.1)).fit(X_hist, y_hist)
print("a perfectly ordinary fitted pipeline:", pipe)

In [ ]:
# one line to run it blind — scaler (1 depth level) + linear (1) + sigmoid (2)
model = pdpg.sklearn.wrap(pipe)
proba = model.predict_proba(X_enc)     # encrypted P(class 1)
proba

In [ ]:
# verify against the plaintext pipeline (the controller could do this)
got = proba.decrypt()
expected = pdpg.approx.sigmoid(pipe.decision_function(X_client))
print("max abs error vs plaintext pipeline:", np.abs(got - expected).max())

## Hard labels are a decision — decisions need the key

In [ ]:
try:
    model.predict(X_enc)
except pdpg.EncryptedOperationError as e:
    print(f"⛔ {e}")

## Encrypted segmentation: KMeans distances

Same pattern: the provider's own KMeans, applied blind. sklearn's
`transform()` returns euclidean distances — that final square root is
impossible under CKKS, so the wrapper offers `transform_squared()`: same
centroid ranking, square root after decryption.

In [ ]:
from sklearn.cluster import KMeans

segmenter = make_pipeline(StandardScaler(), KMeans(n_clusters=3, random_state=0, n_init=10)).fit(X_hist)
dist2 = pdpg.sklearn.wrap(segmenter).transform_squared(X_enc)
dist2

In [ ]:
# the segment ASSIGNMENT happens key-side: decrypt distances, then argmin
segment = dist2.decrypt().argmin(axis=1)
print("segment sizes:", np.bincount(segment))

plain_dist = segmenter.named_steps["kmeans"].transform(
    segmenter.named_steps["standardscaler"].transform(X_client))
print("match plaintext pipeline:", np.array_equal(segment, plain_dist.argmin(axis=1)))

## Notes

- **Inference only.** Fitting happens in plaintext, where the training data
  lives. What runs on ciphertext is the fitted model.
- Supported: linear regressors (multi-output included), binary linear
  classifiers, `StandardScaler`, `KMeans`, and Pipelines of those.
- Multiclass is refused up front: softmax needs `exp` and division.

<sub>A [PDPG-lab](https://pdpglab.xyz) project. Current backend:
[TenSEAL](https://github.com/OpenMined/TenSEAL) (CKKS). More recipes in
[demo/cookbook](https://github.com/PDPG-lab/pypdpg/tree/main/demo/cookbook).</sub>